# Week 6, Lab 2 — Build / extend a local MCP server


In [1]:
import zipfile
import os

zip_path = "/content/shared.zip"      # Path of the uploaded ZIP file
extract_path = "/content/shared"      # Folder where files will be extracted

# Create the folder if it doesn't exist
os.makedirs(extract_path, exist_ok=True)

# Unzip
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ ZIP extracted successfully!")
print("Files extracted to:", extract_path)

✅ ZIP extracted successfully!
Files extracted to: /content/shared


In [2]:
import zipfile
import os

zip_path = "/content/shared.zip"
extract_path = "/content"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Extracted successfully!")

✅ Extracted successfully!


In [3]:
WEEK = 'Week 6'
LAB = 'Lab 2 — build server'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


Week 6 / Lab 2 — build server
Environment: Google Colab
Backend: huggingface
Tip: Runtime → Change runtime type → T4 GPU for faster generation.
If import failed, unzip/clone the WHOLE course folder (not a single notebook).


In [4]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate fastapi uvicorn mcp
else:
    %pip install -q mcp ollama


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 2.7 MB/s eta 0:00:00


In [4]:
%%writefile /content/local_tools_server.py

from mcp.server.fastmcp import FastMCP

mcp = FastMCP("LocalTools")

@mcp.tool()
def calculator(expression: str) -> str:
    """Evaluate an arithmetic expression."""
    return str(eval(expression))

@mcp.tool()
def lookup_fact(topic: str) -> str:
    """Return a local AI fact."""
    facts = {
        "langgraph": "LangGraph builds stateful LLM workflows as graphs of nodes and edges.",
        "ollama": "Ollama lets you run open-source LLMs locally on your computer.",
        "mcp": "Model Context Protocol (MCP) connects AI models to tools and data sources.",
        "crewai": "CrewAI lets multiple AI agents collaborate on tasks."
    }
    return facts.get(topic.lower(), "No fact found.")

if __name__ == "__main__":
    mcp.run()

Overwriting /content/local_tools_server.py


In [5]:
from pathlib import Path

SERVER_PATH = Path("/content/local_tools_server.py")

print("Exists:", SERVER_PATH.exists())
print("Path:", SERVER_PATH)

Exists: True
Path: /content/local_tools_server.py


In [2]:
from mcp.server.fastmcp import FastMCP

# ---------------- Create MCP Server ----------------

mcp = FastMCP("demo")

@mcp.tool()
def shout(text: str) -> str:
    """Convert text to uppercase."""
    return text.upper()

@mcp.tool()
def calculator(expression: str) -> str:
    """Evaluate arithmetic expression."""
    return str(eval(expression))

# ---------------- Call Tools Directly ----------------

print("📌 Available Tools:")
for tool in mcp._tool_manager.list_tools():
    print(f"- {tool.name}: {tool.description}")

print("\n🗣️ Shout Tool Output:")
print(shout("mcp is a protocol"))

print("\n🧮 Calculator Tool Output:")
print(calculator("45*12+30"))

📌 Available Tools:
- shout: Convert text to uppercase.
- calculator: Evaluate arithmetic expression.

🗣️ Shout Tool Output:
MCP IS A PROTOCOL

🧮 Calculator Tool Output:
570


Add your own course topics to the KB in `local_tools_server.py`.
